<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. Clasificación Multiclase — Dibujar cercas en el mapa de las flores
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/02_Clasificacion_Multiclase_y_Fronteras_Decision_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Hasta ahora clasificamos en 2 categorías (Sí/No). Aquí aprenderás a clasificar en **3 o más categorías** y a **visualizar las fronteras de decisión** del modelo:

1. Qué es la **clasificación multiclase** y cómo funciona.
2. Cómo visualizar las **fronteras de decisión** (las cercas del mapa).
3. Qué es la **regularización** y por qué evita que el modelo se "memorice" los datos.

---
## 1. La analogía del mapa de parcelas de flores 🌸

Imagina que tienes un terreno con 3 tipos de flores mezcladas:
- 🔴 **Setosa** — pétalos muy pequeños
- 🟢 **Versicolor** — pétalos medianos
- 🔵 **Virginica** — pétalos grandes

Si graficas cada flor en un mapa donde el eje X es el largo del pétalo y el eje Y es el ancho, verás que las flores del mismo tipo tienden a agruparse.

Un modelo de clasificación es como **dibujar cercas** en ese mapa: divide el terreno en zonas, y cualquier flor nueva que aparezca se clasifica según en qué zona cae.

Esas **cercas** son las **Fronteras de Decisión (*Decision Boundaries*)** del modelo.

In [ ]:
import os, urllib.parse, urllib.request, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

def load_dataset(filename, module_name="08 - Classification"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    return target_path

iris = pd.read_csv(load_dataset('iris.csv'))
print(f"Dataset Iris: {iris.shape[0]} flores · {iris.shape[1]-1} características")
print(f"Especies: {iris['species'].unique()}")
print(iris['species'].value_counts())

In [ ]:
# Visualizar las flores en el mapa 2D (largo vs ancho del pétalo)
plt.figure(figsize=(10, 6))
colores = {'setosa': '#ef4444', 'versicolor': '#10b981', 'virginica': '#6366f1'}
for especie, color in colores.items():
    subset = iris[iris['species'] == especie]
    plt.scatter(subset['petal_length'], subset['petal_width'],
                label=especie, color=color, s=70, alpha=0.8, edgecolors='white')

plt.xlabel('Largo del pétalo (cm)', fontsize=12)
plt.ylabel('Ancho del pétalo (cm)', fontsize=12)
plt.title('🌸 Mapa de flores Iris\n(cada punto = una flor, el color = su especie)', fontweight='bold', fontsize=13)
plt.legend(title='Especie')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Las flores del mismo tipo se agrupan naturalmente.")
print("   El modelo necesita encontrar las 'cercas' que separan cada grupo.")

---
## 2. Entrenando el clasificador multiclase y visualizando las cercas 🖼️

In [ ]:
# Usamos solo 2 variables para poder dibujar el mapa en 2D
X = iris[['petal_length', 'petal_width']].values
le = LabelEncoder()
y = le.fit_transform(iris['species'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

modelo = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42)
modelo.fit(X_train_sc, y_train)

print(f"✅ Exactitud: {accuracy_score(y_test, modelo.predict(X_test_sc)):.1%}")

# Dibujar las fronteras de decisión (las cercas)
h = 0.02
x_min, x_max = X_train_sc[:, 0].min() - 1, X_train_sc[:, 0].max() + 1
y_min, y_max = X_train_sc[:, 1].min() - 1, X_train_sc[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(10, 6))
cmap_fondo = plt.cm.RdYlBu
plt.contourf(xx, yy, Z, alpha=0.25, cmap=cmap_fondo)
plt.contour(xx, yy, Z, colors='black', linewidths=1.5, linestyles='--')

colores_num = ['#ef4444', '#10b981', '#6366f1']
for cls, color, nombre in zip([0, 1, 2], colores_num, le.classes_):
    mask = y_train == cls
    plt.scatter(X_train_sc[mask, 0], X_train_sc[mask, 1],
                color=color, label=nombre, s=60, edgecolors='white', alpha=0.9)

plt.title('🖼️ Fronteras de Decisión del modelo Logístico Multiclase\n(las líneas punteadas son las "cercas" del modelo)',
          fontweight='bold', fontsize=12)
plt.xlabel('Largo del pétalo (estandarizado)')
plt.ylabel('Ancho del pétalo (estandarizado)')
plt.legend(title='Especie')
plt.tight_layout()
plt.show()

---
## 3. Regularización — Evitar que el modelo se memorice los datos 🧠

Imagina que estudias para un examen **memorizando** todas las preguntas del libro de prácticas, pero el examen tiene preguntas nuevas. Memorizaste mucho, pero no aprendiste a pensar.

En ML, eso se llama **sobreajuste (overfitting)**: el modelo "memoriza" los datos de entrenamiento pero falla con datos nuevos.

La **regularización** es la solución: penaliza los modelos que son "demasiado complicados". El parámetro `C` controla esto:
- **C pequeño** (ej. 0.01) → regularización fuerte → modelo más simple, menos sobreajuste.
- **C grande** (ej. 100) → regularización débil → modelo puede volverse muy complejo.

In [ ]:
from sklearn.metrics import accuracy_score

valores_C = [0.01, 0.1, 1.0, 10, 100]
print(f"{'C':>8} | {'Acc. Entrenamiento':>20} | {'Acc. Prueba':>14} | Diagnóstico")
print("-" * 70)

for C in valores_C:
    m = LogisticRegression(C=C, multi_class='multinomial', max_iter=1000, random_state=42)
    m.fit(X_train_sc, y_train)
    acc_train = accuracy_score(y_train, m.predict(X_train_sc))
    acc_test  = accuracy_score(y_test,  m.predict(X_test_sc))
    diff = acc_train - acc_test
    diag = '⚠️ Sobreajuste' if diff > 0.05 else ('✅ Equilibrado' if diff >= 0 else '📉 Subajuste')
    print(f"{C:>8} |       {acc_train:.3f}            |    {acc_test:.3f}     | {diag}")

print("\n💡 El mejor C es aquel donde el acc. de prueba es alto y la diferencia con entrenamiento es pequeña.")

---
## 4. Reporte de clasificación completo 📊

In [ ]:
print("📊 Reporte detallado por especie:")
print(classification_report(y_test, modelo.predict(X_test_sc), target_names=le.classes_))
print("\n💡 Glosario rápido:")
print("   precision = de las que predije como X, ¿cuántas son realmente X?")
print("   recall    = de las que son realmente X, ¿cuántas detecté?")
print("   f1-score  = equilibrio entre precision y recall (1.0 = perfecto)")
print("   support   = cuántas flores de esa especie hay en el conjunto de prueba")

---
## 5. Resumen 🎓

- ✅ **Clasificación multiclase** = predecir entre 3 o más categorías (Softmax internamente).
- ✅ Las **fronteras de decisión** son las "cercas" que separan cada zona del mapa.
- ✅ La **regularización** (parámetro `C`) evita que el modelo se memorice los datos.
- ✅ `C` pequeño = más regularización = modelo más simple. `C` grande = menos regularización.

> 🚀 **Siguiente paso:** Ve al cuaderno `03_Evaluacion_de_Modelos_y_Metricas_Clasificacion_Dummies.ipynb`.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>